# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pathlib
import os
from dotenv import load_dotenv
import isodate
import datetime

def get_isodatetime():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.datetime_isoformat(current)
    return iso_value

def get_isodate():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.date_isoformat(current)
    return iso_value
#local functions
import video_extraction
import audio_transcription
import transcript_merger
import summarize_transcript

# Load environment variables
load_dotenv()
logger = video_extraction.logger

## Transcribe

In [ ]:
# Args
#-- extraction
dir_subfolder="client" #change for subfolder export. "" is none
input_mkv = "./+recordings/2024-09-26_11-57-44-meetingnotes.mkv" 

# -- transcribing
mic_audio_filename = f"microphone_audio.wav"
desktop_audio_filename = f"desktop_audio.wav"
# -- processing/merging
mic_transcript_filename = f"microphone_transcription.txt"
desktop_transcript_filename = f"desktop_transcription.txt"
transcript_filename = f"merged_transcript.json"
# -- summarizing
summary_filename=f"summary.txt"
summarize_model_name = 'claude-3-5-sonnet'

# Script Code
# - Setup output directories
output_dir = f"./{get_isodate()}-output"
output_path = pathlib.Path(output_dir) if dir_subfolder in ["",None,False] else pathlib.Path(output_dir) / dir_subfolder
if not output_path.exists():
    output_path.mkdir(parents=True)
    
print(f"Output dir is: {output_path}")
# -- Extract audio files
try:
   result = video_extraction.process_mkv_file(input_mkv, str(output_path))
except Exception as e:
    logger.error(f"An error occurred during the extraction process: {str(e)}")
    
# -- Transcribe audio
mic_audio_path = output_path / mic_audio_filename
desktop_audio_path = output_path / desktop_audio_filename
try:
    filenames = audio_transcription.process_audio_files(str(mic_audio_path), str(desktop_audio_path), output_dir=str(output_path))
    logger.info(filenames)
except Exception as e:
    logger.error(f"An error occurred during the transcription process: {str(e)}")

# -- Merge and process transcription
mic_transcript_path = filenames['microphone']
desktop_transcript_path = filenames['desktop']
transcript_file_path = output_path / transcript_filename
try:
    transcript_json = transcript_merger.process_transcripts(
        mic_transcript_file=str(mic_transcript_path), 
        desktop_transcript_file=str(desktop_transcript_path), 
        output_file=str(transcript_file_path),
        parse_overlaps=False
    )
except Exception as e:
    logger.error(f"An error occurred during the merging process: {str(e)}")

# -- Augment with additional information
# TODO: add personal notes taken during meeting
# TODO: add other md, txt, or other text files


# -- Summarize
# Main workflow
summary_filepath = output_path / summary_filename

transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text, 
    model=summarize_model_name
)

# summarize=False #True to call the llm
# # if summarize:
# #     try:
#         summary = summarize_transcript.generate_summary(transcript_text, model_name=summarize_model_name)
#         summarize_transcript.save_summary(
#             summary, 
#             output_file=str(summary_filepath)
#         )
#         logger.info(f"Summary generated and saved to {summary_filepath}")
#     except Exception as e:
#         logger.error(f"An error occurred during summarization: {str(e)}")
#     logger.info(summary)
# else:
#     logger.info("Skipping summarize step")

## Summarize

In [ ]:
import pathlib
import os
from dotenv import load_dotenv
import isodate
import datetime

import summarize_transcript
from anthropic import Anthropic
def get_isodatetime():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.datetime_isoformat(current)
    return iso_value

def get_isodate():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.date_isoformat(current)
    return iso_value
#local functions
import video_extraction
import audio_transcription
import transcript_merger
import summarize_transcript

# Load environment variables
load_dotenv()
logger = video_extraction.logger

In [ ]:
def get_filename_prompt(content):
    current_date = get_isodate()
    prompt = f"""Please read the <content> completely and provide a filename that can be used to write the file to disk as a markdown. The filename should be concise but also descriptive so its clear what the content contains.

    <filenamerules>
    - only use lowercase letters, numbers, and underscores or dashes
    - prefer underscores where possible as they are operating system independent
    - make the filename convenient for all operating systems (windows, macOS, linux)
    - include the current date in the filename as YYYMMDD (e.g. July 4th , 2024 would be 20240704)
    - include a concise but descriptive name related to the <content>
    - keep the filename as short as possible while containing a useful name and date
    - DONOT include the suffix of the filename and make sure their are no period characters in the filename
    - the current date is {current_date}
    </filenamerules>

    <content>
    {content}
    </content>

    remember ONLY return the filename as a string and follow the <filenamerules>
    """
    return prompt
     

def get_filename_from_content(content, model_name='claude-3-5-sonnet'):
    """obtain a good filename from content and date"""
    
    anthropic_models = summarize_transcript.anthropic_models
    if model_name not in anthropic_models:
        raise KeyError(f"model must be one of [{list(anthropic_models.keys())}]")
    model = anthropic_models.get(model_name)['name']
    client = Anthropic(api_key=summarize_transcript.ANTHROPIC_API_KEY)
    prompt = get_filename_prompt(content)
    
    try:
        response = client.messages.create(
            model=model,
            max_tokens=100,
            temperature=0.7,
            messages=[
                {"role": "user", "content": prompt}
            ],
        )
        return response.content[0].text
    except Exception as e:
        logger.error(f"Error generating summary: {str(e)}")
        raise

def create_markdown_report(content, output_path):
    """reads content and comes up with a good name, then writes to markdown"""
    filename = get_filename_from_content(content, model_name='claude-3-5-sonnet')
    filepath = output_path / f"{filename}.md"
    logger.info(f"Writing markdown file to {filepath}")
    with open(filepath, 'w', encoding='utf-8') as file:
        file.write(content)
    return filepath


In [ ]:
def generate_meeting_summary_prompt(personal_notes, personal_info):
    formatted_notes = ""
    for note_dict in personal_notes:
        formatted_notes += f"- {note_dict['note']}\n"
        for bullet in note_dict.get('bullets', []):
            formatted_notes += f"  * {bullet}\n"
    
    prompt = f"""# 🧙‍♂️ Comprehensive Meeting Insight Generator with Personal Notes Integration

#CONTEXT: In today's fast-paced professional environment, effective communication of meeting outcomes is crucial. This prompt is designed to transform verbose meeting transcripts into clear, actionable summaries that capture the essence of discussions while highlighting key technical details and decisions. Additionally, it incorporates personal notes to ensure critical observations and points are addressed and emphasized in the summary.

#ROLE: You are an expert meeting analyst and technical communicator with a keen eye for detail. Your task is to distill complex discussions into coherent, structured summaries that provide value to both attendees and non-attendees alike, while ensuring that personal notes and observations are seamlessly integrated and highlighted.

#RESPONSE GUIDELINES:
1. Begin with an executive summary (who, what, where, when)
2. Analyze the transcript and determine the appropriate number of major topics discussed. Create a numbered list of these topics.
3. For each major topic:
   a. Create a heading
   b. Expand on technical content
   c. Describe existing or proposed projects/efforts
   d. Outline decisions made
   e. Specify timelines
   f. Integrate relevant personal notes, highlighting them distinctly
4. Review and add any additional relevant information
5. Conclude with action items and overall meeting outcomes
6. Create a section specifically addressing personal notes not covered in the main topics

#TASK CRITERIA:
- Use markdown formatting for headers, lists, and emphasis
- Adjust the length of the summary based on the meeting's complexity and duration, typically aiming for 2-4 pages of content
- Focus on technical accuracy and clarity
- Ensure the summary is self-contained and understandable without context
- Highlight personal notes using a distinct formatting style (e.g., blockquotes or a specific emoji)
- Prioritize addressing all personal notes, ensuring they are contextually integrated
- The number of major topics should be determined based on the meeting's content and complexity

#INFORMATION ABOUT ME: {personal_info}

#PERSONAL NOTES:
{formatted_notes}

#OUTPUT:
Provide the summary in the following structure:

## Executive Summary

[Brief overview, including any high-priority personal notes]

## Major Topics Discussed

1. [Topic 1]
2. [Topic 2]
3. [Topic 3]
... (continue as needed based on the meeting's content)

## Detailed Discussion

### [Topic 1]

[Technical details, projects, decisions, timelines]

> 📝 Personal Note: [Relevant personal note integrated and highlighted]
> * [Related bullet point]
> * [Related bullet point]

[Additional content addressing the personal note]

### [Topic 2]

[Technical details, projects, decisions, timelines]

> 📝 Personal Note: [Relevant personal note integrated and highlighted]
> * [Related bullet point]
> * [Related bullet point]

[Additional content addressing the personal note]

### [Topic 3]

[Technical details, projects, decisions, timelines]

> 📝 Personal Note: [Relevant personal note integrated and highlighted]
> * [Related bullet point]
> * [Related bullet point]

[Additional content addressing the personal note]

... (continue as needed for additional topics)

## Additional Notes

[Any relevant information not covered in the main sections, including personal notes that didn't fit elsewhere]

## Action Items and Conclusion

[List of action items and overall meeting outcomes, with emphasis on any related personal notes]

## Personal Notes Summary

[A brief section addressing any remaining personal notes that weren't integrated into the main summary, explaining their context and importance]

Please summarize the meeting transcript based on the instructions given above, integrating and highlighting the provided personal notes throughout the summary. Determine the appropriate number of major topics based on the meeting's content and complexity.
"""
    return prompt

def extract_transcript_text(transcript_data):
    """
    Extract text from the JSON transcript, combine it into a single string,
    including speaker and timestamp, and ensuring chronological order.
    """
    transcript_entries = transcript_data["transcript"]
    # Sort entries by start_time to ensure chronological order
    sorted_entries = sorted(transcript_entries, key=lambda x: x["start_time"])
    
    formatted_lines = []
    for entry in sorted_entries:
        speaker = entry["source"].capitalize()
        start_time = f"{entry['start_time']:.2f}"
        text = entry["text"].strip()
        formatted_line = f"[{speaker} {start_time}] {text}"
        formatted_lines.append(formatted_line)
    
    return "\n".join(formatted_lines)

In [ ]:
summarize = True

dir_subfolder="client" #change for subfolder export. "" is none
summary_filename=f"summary.txt"
summarize_model_name = 'claude-3-5-sonnet'
transcript_filename = f"merged_transcript.json"

#file info
output_dir = f"./{get_isodate()}-output"
output_path = pathlib.Path(output_dir) if dir_subfolder in ["",None,False] else pathlib.Path(output_dir) / dir_subfolder
transcript_file_path = output_path / transcript_filename
logger.info(f"Output dirname: {output_path}")
logger.info(f"Transcript filename: {transcript_file_path}")
#system prompt info
personal_info = "Product Manager with a background in data and chemical engineering, planning next steps for our techno economic model (TEM) project and business model comparison."

personal_notes = [
    # <redacted: per-meeting notes>
]

# -- Summarize
summary_filepath = output_path / f"{get_isodatetime()}-{summary_filename}"
logger.info(f"Summary filename: {summary_filepath}")

transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
# Main workflow
system_prompt = generate_meeting_summary_prompt(personal_notes, personal_info)
transcript_text = extract_transcript_text(transcript_data)
# Estimate tokens and price
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text, 
    model=summarize_model_name
)
if summarize:
    logger.info(f"Summarizing notes and transcript now...")
    try:
        summary = summarize_transcript.generate_summary(transcript_text, model_name=summarize_model_name, system_prompt=system_prompt)
        summarize_transcript.save_summary(
            summary, 
            output_file=str(summary_filepath)
        )
        logger.info(f"Summary generated and saved to {summary_filepath}")
        filepath = create_markdown_report(content=summary, output_path=output_path)
    except Exception as e:
        logger.error(f"An error occurred during summarization: {str(e)}")
    logger.info("Completed Summarization!")
else:
    logger.info("Skipping summarize step")
